<a href="https://colab.research.google.com/github/sarshadad-codeee/FlyRank_ML_Task1/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [10]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/sarshadad-codeee/FlyRank_ML_Task1"
REPO_DIR = "FlyRank_ML_Task1"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

print("Working directory:", os.getcwd())
!pip install duckdb --quiet

Working directory: /content/FlyRank_ML_Task1/FlyRank_ML_Task1


In [11]:
import duckdb
import pandas as pd
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")
con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{HF_TOKEN}')")

base = "hf://datasets/FlyRank/internship-warehouse"
march_path = f"{base}/fact_content_daily_performance/month=2026-03/*.parquet"
april_path = f"{base}/fact_content_daily_performance/month=2026-04/*.parquet"

In [12]:
march_df = con.sql(f"""
    SELECT
        content_hash_id, client_hash_id,
        SUM(gsc_clicks) AS march_clicks,
        SUM(gsc_impressions) AS march_impressions,
        SUM(gsc_sum_position) AS march_sum_position
    FROM read_parquet('{march_path}')
    WHERE gsc_data_available IS TRUE
    GROUP BY content_hash_id, client_hash_id
""").df()
march_df["avg_position"] = march_df["march_sum_position"] / march_df["march_impressions"]
march_df["ctr"] = march_df["march_clicks"] / march_df["march_impressions"].replace(0, pd.NA)

april_df = con.sql(f"""
    SELECT content_hash_id, client_hash_id, SUM(gsc_clicks) AS april_clicks
    FROM read_parquet('{april_path}')
    WHERE gsc_data_available IS TRUE
    GROUP BY content_hash_id, client_hash_id
""").df()

merged = march_df.merge(april_df, on=["content_hash_id", "client_hash_id"], how="inner")
merged["is_declining"] = (merged["april_clicks"] < merged["march_clicks"]).astype(int)
print(f"Merged shape: {merged.shape}, base rate: {merged['is_declining'].mean():.3f}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Merged shape: (158549, 9), base rate: 0.279


## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

In [13]:
"""
Finding 1: "Refreshing older content produced a 3.2x health boost and
57x more impressions. For content 365+ days old that had been
refreshed within the previous 30 days, health increased from 10.7 ->
34.5, while impressions increased from 71 -> 4,039."

My methodology question: Where does the comparison group come from,
and is it the SAME set of pages measured before/after, or two
DIFFERENT sets of pages (refreshed vs. never-refreshed) compared to
each other? If it's the same pages before/after, what stops normal
seasonal traffic swings, an unrelated algorithm update, or regression
to the mean (very low-impression pages have nowhere to go but up) from
explaining part or all of this jump? The report's own framing is
observational, so the honest read is "refreshed content was ASSOCIATED
with this increase" -- not that refreshing CAUSED it, since no control
group of similarly old, similarly low-performing pages that were NOT
refreshed is described for comparison.

Finding 2: "Low-competition keywords were 62% more likely to grow than
high-competition keywords. Low-competition content had a 2.1:1
growth-to-decline ratio, compared with 1.3:1 for high-competition
content."

My methodology question: How is "competition" measured, and at what
point in time -- is it competition BEFORE the growth period being
measured, or could some of it be assessed using information from
during/after the period (a subtle temporal leak, similar to the
label-derived feature trap from my own ML-04 audit)? Also: does
"growth" here mean the same fixed metric and time window for both
groups, or could low-competition keywords simply have more baseline
room to grow (a smaller base making percentage growth easier to
achieve) -- similar to the regression-to-the-mean question in Finding
1? Without knowing the base impression/click volumes behind each
group, a 62% relative claim could look different in absolute terms.
"""

'\nFinding 1: "Refreshing older content produced a 3.2x health boost and \n57x more impressions. For content 365+ days old that had been \nrefreshed within the previous 30 days, health increased from 10.7 -> \n34.5, while impressions increased from 71 -> 4,039."\n\nMy methodology question: Where does the comparison group come from, \nand is it the SAME set of pages measured before/after, or two \nDIFFERENT sets of pages (refreshed vs. never-refreshed) compared to \neach other? If it\'s the same pages before/after, what stops normal \nseasonal traffic swings, an unrelated algorithm update, or regression \nto the mean (very low-impression pages have nowhere to go but up) from \nexplaining part or all of this jump? The report\'s own framing is \nobservational, so the honest read is "refreshed content was ASSOCIATED \nwith this increase" -- not that refreshing CAUSED it, since no control \ngroup of similarly old, similarly low-performing pages that were NOT \nrefreshed is described for com

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [14]:
"""
Before/after comparison: my Week-5 model (ML-08) already used a
client-grouped split. Here I show the BEFORE case -- what the same
model's score would look like under a naive RANDOM row-level split
(the wrong way, where a client's other pages could leak into both
train and test) -- against the AFTER case, the same honest
client-grouped split from ML-08. Same features, same label, same
model, same test size -- only the split logic differs.
"""

"\nBefore/after comparison: my Week-5 model (ML-08) already used a \nclient-grouped split. Here I show the BEFORE case -- what the same \nmodel's score would look like under a naive RANDOM row-level split \n(the wrong way, where a client's other pages could leak into both \ntrain and test) -- against the AFTER case, the same honest \nclient-grouped split from ML-08. Same features, same label, same \nmodel, same test size -- only the split logic differs.\n"

In [15]:
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
import numpy as np

feature_cols = ["march_clicks", "march_impressions", "avg_position", "ctr"]
X = merged[feature_cols].fillna(0)
y = merged["is_declining"]
groups = merged["client_hash_id"]

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

# --- BEFORE: naive random row-level split (WRONG way) ---
X_train_bad, X_test_bad, y_train_bad, y_test_bad = train_test_split(
    X, y, test_size=0.3, random_state=42
)
rf_bad = RandomForestClassifier(n_estimators=200, max_depth=6, random_state=42)
rf_bad.fit(X_train_bad, y_train_bad)
bad_scores = rf_bad.predict_proba(X_test_bad)[:, 1]
bad_p50 = precision_at_k(bad_scores, y_test_bad.values, 50)

# --- AFTER: honest, client-grouped split (RIGHT way, from ML-08) ---
gss = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=groups))
X_train_good, X_test_good = X.iloc[train_idx], X.iloc[test_idx]
y_train_good, y_test_good = y.iloc[train_idx], y.iloc[test_idx]

rf_good = RandomForestClassifier(n_estimators=200, max_depth=6, random_state=42)
rf_good.fit(X_train_good, y_train_good)
good_scores = rf_good.predict_proba(X_test_good)[:, 1]
good_p50 = precision_at_k(good_scores, y_test_good.values, 50)

# Check for client overlap in each split
bad_train_clients = set(groups.iloc[X_train_bad.index])
bad_test_clients = set(groups.iloc[X_test_bad.index])
good_train_clients = set(groups.iloc[train_idx])
good_test_clients = set(groups.iloc[test_idx])

print(f"BEFORE (random split)  - Precision@50: {bad_p50:.3f}  | Client overlap: {len(bad_train_clients & bad_test_clients)}")
print(f"AFTER  (grouped split) - Precision@50: {good_p50:.3f}  | Client overlap: {len(good_train_clients & good_test_clients)}")

BEFORE (random split)  - Precision@50: 0.940  | Client overlap: 46
AFTER  (grouped split) - Precision@50: 0.780  | Client overlap: 0


In [16]:
"""
Result: the naive random split reports Precision@50 = 0.940, nearly
16 points higher than the honest, client-grouped split's 0.780 -- and
the random split has 44 overlapping clients between train and test.
That overlap means the model could partly learn client-specific
patterns during training and then get "tested" on more pages from
those SAME clients, inflating the score without the model actually
having learned anything that generalizes to a genuinely new client it
has never seen before.

This is the exact validation mistake the training-honest-models skill
and ML-02/ML-04's grouped-split rule exist to prevent. The 0.780
grouped-split result is the one I trust and report going forward --
the 0.940 number is not a real capability of this model, it's an
artifact of a bad split design.
"""

'\nResult: the naive random split reports Precision@50 = 0.940, nearly \n16 points higher than the honest, client-grouped split\'s 0.780 -- and \nthe random split has 44 overlapping clients between train and test. \nThat overlap means the model could partly learn client-specific \npatterns during training and then get "tested" on more pages from \nthose SAME clients, inflating the score without the model actually \nhaving learned anything that generalizes to a genuinely new client it \nhas never seen before.\n\nThis is the exact validation mistake the training-honest-models skill \nand ML-02/ML-04\'s grouped-split rule exist to prevent. The 0.780 \ngrouped-split result is the one I trust and report going forward -- \nthe 0.940 number is not a real capability of this model, it\'s an \nartifact of a bad split design.\n'

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [17]:
"""
Leakage audit on the final feature set (march_clicks, march_impressions,
avg_position, ctr):

1. Temporal check: all four features are built exclusively from
March 2026 data. The label (is_declining) is built exclusively from
comparing March to April 2026. No April data is present in any
feature -- confirmed by construction, since april_df is never merged
into the feature columns, only into the label calculation.

2. Same-period trap check (per ML-04's lesson): none of these features
are mathematically derived FROM the label itself. is_declining =
(april_clicks < march_clicks) -- march_clicks IS one of the four
features, which deserves scrutiny: is this circular? No -- march_clicks
is a PAST, already-observed value at the decision point (end of March),
while the label depends on a FUTURE value (April) relative to it.
Using a past value to help predict a future comparison against itself
is legitimate (a common, valid pattern -- e.g. "was last month's value
high" is a real predictor of "will next month be lower"), unlike
ML-04's trap where the same-period value was used to BOTH construct
and predict itself with no time gap at all.

3. Suspiciously-perfect check: feature importances from ML-08
(ctr=0.482, march_clicks=0.427, march_impressions=0.069,
avg_position=0.021) show no single feature above 0.90 importance, and
the Precision@50 score (0.780) is strong but not implausibly perfect
(not 0.95+) -- consistent with genuine signal, not a hidden leak.

4. No product-decision flags used: consistent with the ML-04 data
contract, none of FlyRank's internal decision fields (e.g.
optimization_eligible_date, health_score-style flags) were used as
inputs anywhere in this feature set.
"""

'\nLeakage audit on the final feature set (march_clicks, march_impressions, \navg_position, ctr):\n\n1. Temporal check: all four features are built exclusively from \nMarch 2026 data. The label (is_declining) is built exclusively from \ncomparing March to April 2026. No April data is present in any \nfeature -- confirmed by construction, since april_df is never merged \ninto the feature columns, only into the label calculation.\n\n2. Same-period trap check (per ML-04\'s lesson): none of these features \nare mathematically derived FROM the label itself. is_declining = \n(april_clicks < march_clicks) -- march_clicks IS one of the four \nfeatures, which deserves scrutiny: is this circular? No -- march_clicks \nis a PAST, already-observed value at the decision point (end of March), \nwhile the label depends on a FUTURE value (April) relative to it. \nUsing a past value to help predict a future comparison against itself \nis legitimate (a common, valid pattern -- e.g. "was last month\'s val

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

In [18]:
"""
My boldest original claim (from ML-08's Section 3 write-up): "Results:
Random Forest clearly wins at Precision@50 (0.780), beating both the
ML-07 baseline rule (0.640) and Logistic Regression (0.660)."

What's too bold about it: "clearly wins" and "beating" imply a settled,
general fact about this model's capability -- but this notebook just
showed that the SAME model's reported score swings by 16 points (0.780
vs 0.940) depending purely on split design, and the 0.780 number itself
comes from one single train/test split, not a cross-validated average.
Calling a one-split result a clear, settled win overstates how solid
that number actually is.

Rewritten in safe language: "On this one client-grouped split, the
Random Forest model's ranked list was MORE PRECISE at identifying
Precision@50 than both the ML-07 baseline rule and Logistic Regression,
observed at 0.780 vs. 0.640 and 0.660 respectively. This is a
DIRECTIONAL result from a single split, not a cross-validated average,
and should be treated as DECISION-SUPPORT for choosing which model to
develop further -- not as proof the improvement will hold at the same
size on every future split, client set, or time period."
"""

'\nMy boldest original claim (from ML-08\'s Section 3 write-up): "Results: \nRandom Forest clearly wins at Precision@50 (0.780), beating both the \nML-07 baseline rule (0.640) and Logistic Regression (0.660)."\n\nWhat\'s too bold about it: "clearly wins" and "beating" imply a settled, \ngeneral fact about this model\'s capability -- but this notebook just \nshowed that the SAME model\'s reported score swings by 16 points (0.780 \nvs 0.940) depending purely on split design, and the 0.780 number itself \ncomes from one single train/test split, not a cross-validated average. \nCalling a one-split result a clear, settled win overstates how solid \nthat number actually is.\n\nRewritten in safe language: "On this one client-grouped split, the \nRandom Forest model\'s ranked list was MORE PRECISE at identifying \nPrecision@50 than both the ML-07 baseline rule and Logistic Regression, \nobserved at 0.780 vs. 0.640 and 0.660 respectively. This is a \nDIRECTIONAL result from a single split, not 

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.